# B3 · Mô hình nhanh trên vector → FILE NỘP ĐẦU TIÊN  (5–10 phút)

Thử mọi vector × (hồi quy logistic, LightGBM), chấm chéo 5 phần, chọn cái tốt nhất. **Nộp file này trước để có điểm**, rồi mới sang B4.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # thư mục notebooks (chứa b2lib.py)
import numpy as np, pandas as pd
import b2lib as L
print("thư mục làm việc:", L.WORK)

In [ ]:
df = L.doc_bang_chuan()
itr, ite = (df.split == "train").to_numpy(), (df.split == "test").to_numpy()
y = df.loc[itr, "label"].astype(str).to_numpy()
ids_tr, ids_te = df.loc[itr, "id"].to_numpy(), df.loc[ite, "id"].to_numpy()
print(itr.sum(), "train |", ite.sum(), "test |", len(set(y)), "lớp")

In [ ]:
F = dict(np.load(L.WORK / "dac_trung.npz"))
giu = np.load(L.WORK / "giu_lai.npy") if (L.WORK / "giu_lai.npy").exists() else np.ones(itr.sum(), bool)
DUNG_DU_LIEU_SACH = True          # ✏️ False = bỏ qua kết quả làm sạch ở B2
m = giu if DUNG_DU_LIEU_SACH else np.ones_like(giu)
Ftr = {k: v[itr][m] for k, v in F.items()}; Fte = {k: v[ite] for k, v in F.items()}
yy = y[m]
anh = [k for k in F if k.startswith("anh_")]; chu = [k for k in F if k.startswith("chu_")]
if anh and chu:                   # ghép ảnh + chữ
    Ftr["anh+chu"] = np.hstack([Ftr[k] for k in anh + chu]); Fte["anh+chu"] = np.hstack([Fte[k] for k in anh + chu])
if "anh_clip" in F:               # CLIP đoán theo tên nhãn (không học) — chỉ hợp khi nhãn là chữ tiếng Anh
    Ftr["san_clip"], Fte["san_clip"] = L.clip_doan_theo_ten(Ftr["anh_clip"], Fte["anh_clip"], np.unique(yy))
print(list(Ftr), "| train dùng:", len(yy))

## ✏️ Thước đo của đề: `accuracy` (mặc định), `f1_macro`; bài đoán số: `rmse`, `mae`, `r2`

In [ ]:
CHI_SO = None   # vd "f1_macro"
pred, proba, board, best = L.mo_hinh_nhanh(Ftr, Fte, yy, chi_so=CHI_SO)

In [ ]:
FILE_MAU = ""          # ✏️ đường dẫn sample_submission
COT_ID, COT_NHAN = "id", "label"      # ✏️ tên cột trong file nộp mẫu
if proba is not None: L.luu_xac_suat("nhanh", ids_te, proba, proba.columns)
L.ghi_nop(ids_te, pred, proba, FILE_MAU or None, COT_ID, COT_NHAN, ten="nop_nhanh")

---
## (Đề GHÉP ẢNH ↔ CHỮ / tìm ảnh theo câu) — dùng vector CLIP chung không gian

In [ ]:
# img = L.anh_embed(df.loc[ite, "image"], "clip"); txt = L.chu_embed(df.loc[ite, "text"], "clip_text")
# top, S = L.khop_anh_chu(img, txt, top=5)      # top[i] = 5 ảnh hợp nhất với câu thứ i